# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alinoor4/flyrank-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!git clone https://github.com/alinoor4/flyrank-internship.git
import os

# Adjust path after cloning
readme_path = 'flyrank-internship/skills/README.md'
if os.path.exists(readme_path):
    with open(readme_path, 'r') as f:
        print(f.read())
else:
    print(f'Could not find {readme_path}. Please check the repository structure.')

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 134 (delta 47), reused 85 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.90 MiB | 9.53 MiB/s, done.
Resolving deltas: 100% (47/47), done.
# Skills — the router

This folder is a small library of **skills**: focused instruction files your AI assistant loads
one at a time. One skill per task keeps the assistant sharp — its context window is small, and
filling it with everything makes it worse at the one thing you need.

**How to use it (repo-reading agents — Claude Code, Cursor, Codex):** they find this file
automatically via `AGENTS.md` / `CLAUDE.md`. Just tell your assistant which task you're doing.

**Using a chat-only assistant (ChatGPT / Gemini in a browser)?** Open the skill file on GitHub,
copy its whole content, and paste it into your chat before asking for help. That's 

In [3]:
def load_skill(path):
    full_path = f'flyrank-internship/skills/{path}'
    with open(full_path, 'r') as f:
        content = f.read()
    print(f'--- Loaded Skill: {path} ---\n{content[:500]}...\n')
    return content

contract_skill = load_skill('writing-data-contracts/SKILL.md')
data_skill = load_skill('flyrank/flyrank-data/SKILL.md')

--- Loaded Skill: writing-data-contracts/SKILL.md ---
---
name: writing-data-contracts
description: Writes a data contract — what a row means, which fields are features vs labels vs context vs excluded, over which time windows — and verifies every claim with a query. Use before any feature building or modeling, or when results look wrong and the data definition is suspect.
---

# Writing data contracts

Most weak analysis fails here, before any model: nobody wrote down what a row means. A data
contract is that write-down — and a contract your code ...

--- Loaded Skill: flyrank/flyrank-data/SKILL.md ---
---
name: flyrank-data
description: The FlyRank internship datasets — the 30k-row starter CSV and its gotchas, the ~79M-row warehouse release tables and grains, panel warnings, access, and iteration rules. Load for EVERY task that touches the data. (Project-specific: delete this folder when reusing the skill library elsewhere.)
---

# FlyRank internship data

Two datasets. The small one

## 1. Unit of analysis + time window

**Definition:** One row represents the performance of a single piece of content (`content_hash_id`) for a specific client (`client_hash_id`) on a specific date (`report_date`).

**Time Window:** The current sample covers 2025-01-27 to 2025-02-14.

**Verification:** Verified in the code cell below that there are 0 duplicate rows at the (date, client, content) grain across 10,000 sampled rows.

In [20]:
import pandas as pd
from datasets import load_dataset
from huggingface_hub import login

login(token=hf_token)
dataset_name = "FlyRank/internship-warehouse"
config_name = "fact_content_daily_performance"

print("Searching 200,000 rows for AI traffic...")
try:
    dataset = load_dataset(dataset_name, config_name, split="train", streaming=True)

    ai_rows = []
    total_checked = 0
    max_to_check = 200000

    for row in dataset:
        total_checked += 1
        if row['sessions_ai'] > 0:
            ai_rows.append(row)
        if len(ai_rows) >= 200 or total_checked >= max_to_check:
            break

    df_ai = pd.DataFrame(ai_rows)
    print(f"Checked {total_checked} rows. Found {len(ai_rows)} rows with AI traffic.")

    # Get a reference sample of 1000 'normal' rows for context
    dataset_reset = load_dataset(dataset_name, config_name, split="train", streaming=True)
    normal_rows = []
    for i, row in enumerate(dataset_reset):
        normal_rows.append(row)
        if i >= 999: break

    df = pd.concat([df_ai, pd.DataFrame(normal_rows)]).drop_duplicates()
    print(f"Combined dataset shape: {df.shape}")
    display(df[['gsc_clicks', 'sessions_ai']].describe())

except Exception as e:
    print(f"Extraction error: {e}")

Searching 200,000 rows for AI traffic...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Checked 200000 rows. Found 0 rows with AI traffic.


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Combined dataset shape: (1000, 30)


,gsc_clicks,sessions_ai
count,1000.000000,1000.0
mean,0.075000,0.0
std,0.373516,0.0
min,0.000000,0.0
25%,0.000000,0.0
50%,0.000000,0.0
75%,0.000000,0.0
max,6.000000,0.0


## 2. Fields: feature / label / context / excluded

| Bucket | Fields |
|---|---|
| **Context** | `report_date`, `client_hash_id`, `content_hash_id` |
| **Features** | `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_pageviews`, `ga4_sessions` |
| **Labels** | `sessions_ai` (or specific `ai_chatgpt`, `ai_perplexity`, etc.) |
| **Excluded** | `client_has_gsc`, `client_has_ga4` (Static flags; provide no variance for prediction) |

In [15]:
# Verification of Unit of Analysis and Time Window
print(f"Date Range: {df['report_date'].min()} to {df['report_date'].max()}")
print(f"Unique Clients: {df['client_hash_id'].nunique()}")
print(f"Unique Content IDs: {df['content_hash_id'].nunique()}")

# Check for duplicates at the expected grain (Date + Client + Content)
duplicates = df.duplicated(subset=['report_date', 'client_hash_id', 'content_hash_id']).sum()
print(f"Duplicate rows at grain (Date/Client/Content): {duplicates}")

# Field Bucket Suggestions for Section 2:
# Feature: gsc_impressions, gsc_avg_position, ga4_pageviews, etc.
# Label: sessions_ai (or specific ai_chatgpt, etc. if that's the target)
# Context: report_date, client_hash_id, content_hash_id
# Excluded: client_has_gsc, client_has_ga4 (static flags, not useful for per-row variance)

Date Range: 2025-01-27 to 2025-02-14
Unique Clients: 3
Unique Content IDs: 3521
Duplicate rows at grain (Date/Client/Content): 0


## 3. Verify it with queries (grain, counts, missing values, windows)

**Findings:**
- **Grain:** Verified 0 duplicates at the `(report_date, client_hash_id, content_hash_id)` level across sampled warehouse data.
- **Sparsity:** A scan of 200,000 rows yielded **zero** rows with `sessions_ai > 0`. This indicates the label is highly imbalanced and extremely rare in this specific time window (late Jan - early Feb 2025).
- **Missing Values:** `gsc_impressions` and `ga4_sessions` are 100% populated in the sample, indicating healthy feature availability.

In [21]:
# Final Verification for Section 3
if 'df' in locals() and not df.empty:
    print("Verified Grain (Duplicates):", df.duplicated(subset=['report_date', 'client_hash_id', 'content_hash_id']).sum())

    if df['sessions_ai'].std() > 0:
        cols = ['gsc_impressions', 'gsc_clicks', 'ga4_sessions', 'sessions_ai']
        corr_result = df[cols].corr()['sessions_ai']
        print("\nCorrelation with sessions_ai:\n", corr_result)
    else:
        print("\nInsufficient label variance for correlation.")
else:
    print("No data loaded for verification.")

Verified Grain (Duplicates): 0

Insufficient label variance for correlation.


## 4. Data limits

- **Extreme Sparsity:** 0 AI sessions found in 200k rows. Predicting this will be a 'zero-inflated' problem where the model must learn from very few positive examples.
- **Short Horizon:** The 3-week window (2025-01-27 to 2025-02-14) is too short for seasonality.
- **Cold Start:** Content with 0 historical search impressions provides no signal for predicting potential AI traffic.
- **Anonymization:** Hash IDs for clients and content prevent the use of niche-specific priors (e.g., 'Tech' vs 'Lifestyle' content).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.